# U02 · 数学练习

做完 `lesson.ipynb` 再来做这个。  
**4 道题，全部手动推导 + 代码验证**。

## 练习 2.1 · 矩阵维度推理

给定 4 个矩阵，**不运行代码，先纸笔推断**下面哪些乘法合法，结果形状是多少。

```
A: (3, 5)
B: (5, 2)
C: (3, 2)
D: (2, 3)
```

判断并写出结果形状（或标注「❌ 非法」）：
1. `A @ B` (3,2)
2. `A @ C` 非法
3. `A @ D` 非法
4. `B @ D` (5,3)
5. `D @ A` (2,5)
6. `(A @ B) @ D` (3,3)

**先写答案，再跑下一个 cell 验证。**

In [8]:
import numpy as np

A = np.ones((3, 5))
B = np.ones((5, 2))
C = np.ones((3, 2))
D = np.ones((2, 3))

# TODO: 逐个取消注释验证
# print('1. A@B =', (A @ B).shape)
# print('2. A@C =', (A @ C).shape)
# print('3. A@D =', (A @ D).shape)
# print('4. B@D =', (B @ D).shape)
# print('5. D@A =', (D @ A).shape)
print('6. (A@B)@D =', ((A @ B) @ D).shape)

6. (A@B)@D = (3, 3)


## 练习 2.2 · 手动实现矩阵乘法

**不使用 `@` 和 `np.matmul`**，用三重循环实现矩阵乘法。

这题的目的是让你**完全理解** $C_{ij} = \sum_k A_{ik} B_{kj}$。

In [13]:
import numpy as np

def matmul(A, B):
    """
    手动实现矩阵乘法
    A: shape (m, n)
    B: shape (n, p)
    返回 C: shape (m, p)
    """
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, f'维度不匹配: A的列数 {n} 必须等于 B 的行数 {n2}'

    C = np.zeros((m, p))
    # TODO: 三重循环
    # 外两层遍历 C 的每个位置 (i, j)
    for i in range(m):
        for j in range(p):
            for k in range(n):
                C[i,j] += A[i,k] * B[k, j]

    # 最内层累加 A[i,k] * B[k,j]
    
    return C

# 测试
A = np.array([[1, 2, 3], [4, 5, 6]], dtype=float)
B = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)

my_result = matmul(A, B)
true_result = A @ B

print('我的结果:\n', my_result)
print('正确结果:\n', true_result)
print('一致?', np.allclose(my_result, true_result))

我的结果:
 [[ 4.  5.]
 [10. 11.]]
正确结果:
 [[ 4.  5.]
 [10. 11.]]
一致? True


## 练习 2.3 · 偏导数推导

给定函数：
$$ f(x, y) = 3x^2 y + 2xy^2 + 5 $$

**纸笔推导**：
1. $\frac{\partial f}{\partial x} = ?$
2. $\frac{\partial f}{\partial y} = ?$

然后在下面 cell 填入你推导的解析式，代码会用数值微分验证你对不对。

In [15]:
def f(x, y):
    return 3*x**2*y + 2*x*y**2 + 5

# TODO: 把你推导的解析式写在这里
def df_dx(x, y):
    # 把 y 当常数，对 x 求导
    h = 1e-5
    # return (f(x+h,y) - f(x,y)) / h
    return 6 * x * y + 2 * y ** 2

def df_dy(x, y):
    # 把 x 当常数，对 y 求导
    h = 1e-5
    # return (f(x,y+h) - f(x,y)) / h
    return 3 * x ** 2 + 4 * x * y

# 数值验证
def numerical_partial(f, idx, x, y, h=1e-5):
    if idx == 0:
        return (f(x+h, y) - f(x, y)) / h
    else:
        return (f(x, y+h) - f(x, y)) / h

for x, y in [(1.0, 2.0), (3.0, -1.0), (0.5, 0.5)]:
    print(f'(x,y)=({x},{y})')
    print(f'  你的 ∂f/∂x = {df_dx(x,y)},  数值验证 = {numerical_partial(f, 0, x, y):.4f}')
    print(f'  你的 ∂f/∂y = {df_dy(x,y)},  数值验证 = {numerical_partial(f, 1, x, y):.4f}')

(x,y)=(1.0,2.0)
  你的 ∂f/∂x = 20.0,  数值验证 = 20.0001
  你的 ∂f/∂y = 11.0,  数值验证 = 11.0000
(x,y)=(3.0,-1.0)
  你的 ∂f/∂x = -16.0,  数值验证 = -16.0000
  你的 ∂f/∂y = 15.0,  数值验证 = 15.0001
(x,y)=(0.5,0.5)
  你的 ∂f/∂x = 2.0,  数值验证 = 2.0000
  你的 ∂f/∂y = 1.75,  数值验证 = 1.7500


## 练习 2.4 · 链式法则 · 2 层网络反向传播 🌟

这道题最重要，**U7 学 GRU 反传就靠它打底**。

前向过程：
$$
\begin{aligned}
z_1 &= x \cdot w_1 + b_1 \\
a_1 &= \sigma(z_1) \quad \text{(sigmoid: } \sigma(u) = \frac{1}{1+e^{-u}} \text{)} \\
z_2 &= a_1 \cdot w_2 + b_2 \\
L &= \frac{1}{2}(z_2 - y)^2
\end{aligned}
$$

**sigmoid 的导数性质（记住）**：$\sigma'(u) = \sigma(u)(1 - \sigma(u)) = a_1(1 - a_1)$

**任务**：用链式法则求所有 4 个参数的梯度：
$$
\frac{\partial L}{\partial w_1},\ \frac{\partial L}{\partial b_1},\ \frac{\partial L}{\partial w_2},\ \frac{\partial L}{\partial b_2}
$$

**提示** — 先写出每个中间量的局部导数：
- $\partial L / \partial z_2 = ?$
- $\partial z_2 / \partial a_1 = ?$
- $\partial a_1 / \partial z_1 = ?$
- $\partial z_1 / \partial w_1 = ?$, $\partial z_1 / \partial b_1 = ?$
- $\partial z_2 / \partial w_2 = ?$, $\partial z_2 / \partial b_2 = ?$

然后链起来。

In [17]:
import numpy as np

def sigmoid(u):
    return 1 / (1 + np.exp(-u))

# 固定一组输入和参数
x = 2.0
w1, b1 = 0.5, 0.1
w2, b2 = -0.8, 0.2
y_true = 1.0

# 前向
z1 = x * w1 + b1
a1 = sigmoid(z1)
z2 = a1 * w2 + b2
L = 0.5 * (z2 - y_true) ** 2
print(f'前向: z1={z1:.4f}, a1={a1:.4f}, z2={z2:.4f}, L={L:.4f}')

# TODO: 反向传播 —— 填入你的链式法则推导
dL_dz2 = z2 - y_true      # TODO: 提示 L = 0.5(z2-y)^2
dL_da1 = dL_dz2 * w2     # TODO: = dL_dz2 * (dz2/da1)
dL_dz1 = dL_da1 * sigmoid(z1) * (1 - sigmoid(z1))     # TODO: = dL_da1 * sigmoid'(z1)

dL_dw1 = dL_dz1 * x       # TODO
dL_db1 = dL_dz1      # TODO
dL_dw2 = dL_dz2 * a1      # TODO
dL_db2 = dL_dz2      # TODO

print(f'dL/dw1 = {dL_dw1}')
print(f'dL/db1 = {dL_db1}')
print(f'dL/dw2 = {dL_dw2}')
print(f'dL/db2 = {dL_db2}')

# ======== 数值微分验证 ========
def forward(w1, b1, w2, b2):
    z1 = x * w1 + b1
    a1 = sigmoid(z1)
    z2 = a1 * w2 + b2
    return 0.5 * (z2 - y_true) ** 2

h = 1e-6
print('\n=== 数值验证（应该与上面接近）===')
print(f'dL/dw1 ≈ {(forward(w1+h,b1,w2,b2) - forward(w1,b1,w2,b2))/h:.6f}')
print(f'dL/db1 ≈ {(forward(w1,b1+h,w2,b2) - forward(w1,b1,w2,b2))/h:.6f}')
print(f'dL/dw2 ≈ {(forward(w1,b1,w2+h,b2) - forward(w1,b1,w2,b2))/h:.6f}')
print(f'dL/db2 ≈ {(forward(w1,b1,w2,b2+h) - forward(w1,b1,w2,b2))/h:.6f}')

前向: z1=1.1000, a1=0.7503, z2=-0.4002, L=0.9803
dL/dw1 = 0.4197709122076004
dL/db1 = 0.2098854561038002
dL/dw2 = -1.0505202653141719
dL/db2 = -1.4002080844760942

=== 数值验证（应该与上面接近）===
dL/dw1 ≈ 0.419771
dL/db1 ≈ 0.209885
dL/dw2 ≈ -1.050520
dL/db2 ≈ -1.400208


---
## ✅ 过关标准

- [ ] 2.1 至少 5/6 题判断正确
- [ ] 2.2 手动 matmul 通过测试
- [ ] 2.3 两个偏导与数值验证一致
- [ ] 2.4 **所有 4 个梯度与数值验证一致** ← 最关键

全部完成后告诉我「U2 做完了」，我会：
1. Review 你的代码和推导
2. 出复述关卡（3 个原理问题）
3. 进入 **U3 · NumPy + 张量思维**